# Building a boolean network from a knowledge graph

This notebook shows how we can construct a boolean network from a knowledge graph, and how we can combine the boolean network with other boolean networks.


In [1]:
import KGBN

## load_signor_network

[SigNOR](https://signor.uniroma2.it/) is a knowledge graph that contains a variety of relations between proteins and phenotypes. Specifically, we are interested in repression and activation relations among proteins.

The `load_signor_network` function is used to generate a string representing a boolean network, as well as a list of relations between genes, given a list of gene names. It works by identifying an approximately minimal Steiner tree for the given genes (represented as proteins in SigNOR), and then creating a subgraph containing all genes in this tree. Then, the 


In [2]:
gene_list = ['CDKN2A', 'MYC', 'TP53']
bn_string, relations = KGBN.load_signor_network(gene_list)

number of genes found: 3
[1029, 4609, 7157]

Edge found among CDKN2A, MYC, TP53, CDK2:
CDK2 down-regulates CDKN2A
CDK2 up-regulates activity TP53
CDK2 up-regulates CDK2
MYC down-regulates quantity by repression CDKN2A
MYC down-regulates quantity by repression CDKN2A
CDK2 up-regulates quantity by stabilization MYC
TP53 up-regulates quantity by stabilization CDKN2A


In [3]:
print(bn_string)

CDK2 = (CDK2) & (! CDKN2A) # Scores: CDK2_activate:0.2; CDKN2A_inhibit:0.48
CDKN2A = (! MYC) # Scores: MYC_inhibit:0.765
MYC = (CDK2) # Scores: CDK2_activate:0.747
TP53 = (CDK2) # Scores: CDK2_activate:0.871


In [4]:
relations

[('CDK2', 'CDK2', 'activate', 0.2),
 ('CDKN2A', 'CDK2', 'inhibit', 0.48),
 ('CDK2', 'TP53', 'activate', 0.871),
 ('CDK2', 'MYC', 'activate', 0.747),
 ('MYC', 'CDKN2A', 'inhibit', 0.765)]

In [ ]:
CDKN2A = (! MYC)
MYC = MYC
TP53 = (CDKN2A)

Every edge in SigNOR is associated with a confidence score. The `score_cutoff` indicates the minimum score necessary for inclusion in the boolean network.

In [4]:
bn_string, relations = KGBN.load_signor_network(['TNFAIP3', 'CCND1', 'MACF1'], score_cutoff = 0.5)
print(bn_string)

Applied score cutoff 0.5, filtered to 19342/40940 edges
number of genes found: 3
[7128, 595, 23499]
CCND1 = (CTNNB1) # Scores: CTNNB1_activate:0.8
CTNNB1 = (! SRC) # Scores: SRC_inhibit:0.76
SRC = SRC
TNFAIP3 = TNFAIP3
TRAF6 = (SRC) & (! TNFAIP3) # Scores: SRC_activate:0.566; TNFAIP3_inhibit:0.701


### Joiner functions

There are several "joiner" functions.

For more on the "inhibitor wins" joiner, see [LM-Merger: a workflow for merging logical models with an application to gene regulatory network models](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-025-06212-2)

In [5]:
bn_string, relations = KGBN.load_signor_network(['TNFAIP3', 'CCND1', 'MACF1'], joiner='majority')
print(bn_string)

number of genes found: 3
[7128, 595, 23499]
CCND1 = 0 # Scores: GSK3B_inhibit:0.783
GSK3B = (GSK3B & !MACF1) # Scores: GSK3B_activate:0.2; MACF1_inhibit:0.436
MACF1 = 0 # Scores: GSK3B_inhibit:0.436
TNFAIP3 = TNFAIP3
TRAF6 = (TRAF6 & !GSK3B & !TNFAIP3) # Scores: GSK3B_inhibit:0.48; TRAF6_activate:0.2; TNFAIP3_inhibit:0.701


In [6]:
bn_string, relations = KGBN.load_signor_network(['KRAS', 'GNAS', 'TP53', 'SMAD4', 'CDKN2A', 'RNF43'], joiner='&')
print(bn_string)

number of genes found: 6
[3845, 2778, 7157, 4089, 1029, 54894]
CDKN2A = (! MYC) # Scores: MYC_inhibit:0.765
GNAS = (! MDM2) # Scores: MDM2_inhibit:0.395
GSK3B = (GSK3B) # Scores: GSK3B_activate:0.2
KRAS = (SRC) # Scores: SRC_activate:0.656
MDM2 = (TP53) # Scores: TP53_activate:0.968
MYC = (! GSK3B) & (! SMAD4) # Scores: GSK3B_inhibit:0.719; SMAD4_inhibit:0.638
RNF43 = RNF43
SMAD4 = (! GSK3B) # Scores: GSK3B_inhibit:0.397
SRC = (GNAS) & (GSK3B) & (SRC) # Scores: GNAS_activate:0.506; GSK3B_activate:0.383; SRC_activate:0.2
TP53 = (! MDM2) & (GSK3B) & (! SRC) & (! RNF43) # Scores: MDM2_inhibit:0.968; GSK3B_activate:0.727; SRC_inhibit:0.524; RNF43_inhibit:0.452


In [7]:
bn_string, relations = KGBN.load_signor_network(['KRAS', 'GNAS', 'TP53', 'SMAD4', 'CDKN2A', 'RNF43'], joiner='&', score_cutoff = 0.5)
print(bn_string)

number of genes found: 6
[3845, 2778, 7157, 4089, 1029, 54894]
CDKN2A = (! MYC) # Scores: MYC_inhibit:0.765
GNAS = GNAS
KRAS = (SRC) # Scores: SRC_activate:0.656
MAPK1 = MAPK1
MYC = (MAPK1) & (! SMAD4) # Scores: MAPK1_activate:0.733; SMAD4_inhibit:0.638
SMAD4 = (MAPK1) # Scores: MAPK1_activate:0.511
SRC = (GNAS) # Scores: GNAS_activate:0.506
TP53 = (! SRC) & (MAPK1) # Scores: SRC_inhibit:0.524; MAPK1_activate:0.777


In [17]:
bn_string, relations = KGBN.load_signor_network(['KRAS', 'GNAS', 'TP53', 'SMAD4', 'CDKN2A', 'RNF43'], joiner='|')
print(bn_string)

number of genes found: 6
[3845, 2778, 7157, 4089, 1029, 54894]
CDKN2A = (! MYC) # Scores: MYC_inhibit:0.765
GNAS = (! MDM2) # Scores: MDM2_inhibit:0.395
GSK3B = (GSK3B) # Scores: GSK3B_activate:0.2
KRAS = (SRC) # Scores: SRC_activate:0.656
MDM2 = (TP53) # Scores: TP53_activate:0.968
MYC = (! GSK3B) | (! SMAD4) # Scores: GSK3B_inhibit:0.719; SMAD4_inhibit:0.638
RNF43 = RNF43
SMAD4 = (! GSK3B) # Scores: GSK3B_inhibit:0.397
SRC = (GNAS) | (GSK3B) | (SRC) # Scores: GNAS_activate:0.506; GSK3B_activate:0.383; SRC_activate:0.2
TP53 = (! MDM2) | (GSK3B) | (! SRC) | (! RNF43) # Scores: MDM2_inhibit:0.968; GSK3B_activate:0.727; SRC_inhibit:0.524; RNF43_inhibit:0.452


In [18]:
bn_string, relations = KGBN.load_signor_network(['KRAS', 'GNAS', 'TP53', 'SMAD4', 'CDKN2A', 'RNF43'],
                                                            joiner='inhibitor_wins')
print(bn_string)

number of genes found: 6
[3845, 2778, 7157, 4089, 1029, 54894]
CDKN2A = !MYC # Scores: MYC_inhibit:0.765
GNAS = !MDM2 # Scores: MDM2_inhibit:0.395
GSK3B = GSK3B # Scores: GSK3B_activate:0.2
KRAS = SRC # Scores: SRC_activate:0.656
MDM2 = TP53 # Scores: TP53_activate:0.968
MYC = (!GSK3B & !SMAD4) # Scores: GSK3B_inhibit:0.719; SMAD4_inhibit:0.638
RNF43 = RNF43
SMAD4 = !GSK3B # Scores: GSK3B_inhibit:0.397
SRC = (GNAS | GSK3B | SRC) # Scores: GNAS_activate:0.506; GSK3B_activate:0.383; SRC_activate:0.2
TP53 = (!MDM2 & !SRC & !RNF43) & GSK3B # Scores: MDM2_inhibit:0.968; GSK3B_activate:0.727; SRC_inhibit:0.524; RNF43_inhibit:0.452


In [19]:
relations

[('MDM2', 'GNAS', 'inhibit', 0.395),
 ('TP53', 'MDM2', 'activate', 0.968),
 ('GSK3B', 'GSK3B', 'activate', 0.2),
 ('GNAS', 'SRC', 'activate', 0.506),
 ('GSK3B', 'SRC', 'activate', 0.383),
 ('SRC', 'SRC', 'activate', 0.2),
 ('MDM2', 'TP53', 'inhibit', 0.968),
 ('GSK3B', 'TP53', 'activate', 0.727),
 ('SRC', 'TP53', 'inhibit', 0.524),
 ('RNF43', 'TP53', 'inhibit', 0.452),
 ('GSK3B', 'MYC', 'inhibit', 0.719),
 ('SMAD4', 'MYC', 'inhibit', 0.638),
 ('MYC', 'CDKN2A', 'inhibit', 0.765),
 ('SRC', 'KRAS', 'activate', 0.656),
 ('GSK3B', 'SMAD4', 'inhibit', 0.397)]

In [20]:
bn_string, relations = KGBN.load_signor_network(['KRAS', 'GNAS', 'TP53', 'SMAD4', 'CDKN2A', 'RNF43'],
                                                            joiner='majority')
print(bn_string)

number of genes found: 6
[3845, 2778, 7157, 4089, 1029, 54894]
CDKN2A = 0 # Scores: MYC_inhibit:0.765
GNAS = 0 # Scores: MDM2_inhibit:0.395
GSK3B = (GSK3B) # Scores: GSK3B_activate:0.2
KRAS = (SRC) # Scores: SRC_activate:0.656
MDM2 = (TP53) # Scores: TP53_activate:0.968
MYC = 0 # Scores: GSK3B_inhibit:0.719; SMAD4_inhibit:0.638
RNF43 = RNF43
SMAD4 = 0 # Scores: GSK3B_inhibit:0.397
SRC = (GNAS & !GSK3B & !SRC) | (!GNAS & GSK3B & !SRC) | (GNAS & GSK3B & !SRC) | (!GNAS & !GSK3B & SRC) | (GNAS & !GSK3B & SRC) | (!GNAS & GSK3B & SRC) | (GNAS & GSK3B & SRC) # Scores: GNAS_activate:0.506; GSK3B_activate:0.383; SRC_activate:0.2
TP53 = (GSK3B & !MDM2 & !SRC & !RNF43) # Scores: MDM2_inhibit:0.968; GSK3B_activate:0.727; SRC_inhibit:0.524; RNF43_inhibit:0.452


In [21]:
bn_string, relations = KGBN.load_signor_network(['KRAS', 'GNAS', 'TP53', 'SMAD4', 'CDKN2A', 'RNF43'],
                                                            joiner='plurality')
print(bn_string)

number of genes found: 6
[3845, 2778, 7157, 4089, 1029, 54894]
CDKN2A = (!MYC) # Scores: MYC_inhibit:0.765
GNAS = (!MDM2) # Scores: MDM2_inhibit:0.395
GSK3B = (!GSK3B) | (GSK3B) # Scores: GSK3B_activate:0.2
KRAS = (!SRC) | (SRC) # Scores: SRC_activate:0.656
MDM2 = (!TP53) | (TP53) # Scores: TP53_activate:0.968
MYC = (!GSK3B & !SMAD4) # Scores: GSK3B_inhibit:0.719; SMAD4_inhibit:0.638
RNF43 = RNF43
SMAD4 = (!GSK3B) # Scores: GSK3B_inhibit:0.397
SRC = (!GNAS & !GSK3B & !SRC) | (GNAS & !GSK3B & !SRC) | (!GNAS & GSK3B & !SRC) | (GNAS & GSK3B & !SRC) | (!GNAS & !GSK3B & SRC) | (GNAS & !GSK3B & SRC) | (!GNAS & GSK3B & SRC) | (GNAS & GSK3B & SRC) # Scores: GNAS_activate:0.506; GSK3B_activate:0.383; SRC_activate:0.2
TP53 = (!GSK3B & !MDM2 & !SRC & !RNF43) | (GSK3B & !MDM2 & !SRC & !RNF43) | (GSK3B & MDM2 & !SRC & !RNF43) | (GSK3B & !MDM2 & SRC & !RNF43) | (GSK3B & !MDM2 & !SRC & RNF43) # Scores: MDM2_inhibit:0.968; GSK3B_activate:0.727; SRC_inhibit:0.524; RNF43_inhibit:0.452


## TCT KG

In [13]:
import importlib
importlib.reload(KGBN.KG_BN)
from KGBN.KG_BN import load_translator_network

In [1]:
gene_list = ['CDKN2A', 'MYC', 'TP53']
type='gene'
joiner='&'
n_hop=2
# sele_predicate=['biolink:regulates']
sele_predicate=None
sele_API=['CATRAX Pharmacogenomics KP - TRAPI 1.5.0']
sele_KG=['SIGNOR']
sele_tax_id=9606

from TCT import name_resolver, translator_metakg, translator_kpinfo, translator_query, TCT
import igraph as ig

if ' ' not in joiner and joiner in ('&', '|'):
    joiner = ' ' + joiner + ' '

category_map = {
    'gene': 'biolink:Gene',
    'protein': 'biolink:Protein',
    'chemical': 'biolink:ChemicalSubstance',
    'drug': 'biolink:Drug',
    'disease': 'biolink:DiseaseOrPhenotypicFeature',
    'phenotype': 'biolink:PhenotypicFeature',
}
input_node_category = category_map.get(type)
if input_node_category is None:
    raise ValueError(f"Unknown type '{type}'. Must be one of: {list(category_map.keys())}")

# 1. Resolve gene symbols → CURIEs
print("Resolving gene names...")
if sele_tax_id:
    sele_tax_id = "NCBITaxon:" + str(sele_tax_id)
    print(f"Selected tax id: {sele_tax_id}")
else:
    print("No tax id selected, using all")
input_info = name_resolver.batch_lookup(gene_list, only_taxa = sele_tax_id)
input_curies = []
curie_to_symbol = {}
for g in gene_list:
    info = input_info.get(g)
    print(f"info for {g}: {info}")
    if info is not None and hasattr(info, 'curie') and info.curie:
        input_curies.append(info.curie)
        curie_to_symbol[info.curie] = g
print(f"Resolved {len(input_curies)}/{len(gene_list)} genes to CURIEs")
if len(input_curies) < 2:
    print("Need at least 2 resolved genes to build a network.")

# 2. Load Translator resources & determine available predicates/APIs
print("Loading Translator resources...")
_, APInames = translator_kpinfo.get_translator_kp_info()
metaKG = translator_metakg.get_KP_metadata(APInames)
APInames, metaKG = translator_metakg.add_plover_API(APInames, metaKG)

sub_cat = [input_node_category]
obj_cat = [input_node_category]
all_predicates = list(set(
    TCT.select_concept(sub_list=sub_cat, obj_list=obj_cat, metaKG=metaKG)))
all_APIs = TCT.select_API(sub_list=sub_cat, obj_list=obj_cat, metaKG=metaKG)

query_APIs = all_APIs
if sele_API is not None:
    filtered_APIs = [a for a in all_APIs if a in sele_API]
    if filtered_APIs:
        query_APIs = filtered_APIs
    else:
        print(f"Warning: none of sele_API found. Using all {len(all_APIs)} available APIs.")

API_predicates = {}
for api in set(metaKG['API']):
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

# 3. Query Translator
def _do_query(curies):
    qj = TCT.format_query_json(curies, [], sub_cat, obj_cat, all_predicates)
    return translator_query.parallel_api_query(
        query_json=qj, select_APIs=query_APIs,
        APInames=APInames, API_predicates=API_predicates,
        max_workers=len(query_APIs))

print(f"Querying Translator with {len(input_curies)} input nodes...")
all_results = _do_query(input_curies)
print(f"Query returned {len(all_results)} results")


Resolving gene names...
Selected tax id: NCBITaxon:9606
info for CDKN2A: TranslatorNode(curie='NCBIGene:1029', label='CDKN2A', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9606'])
info for MYC: TranslatorNode(curie='NCBIGene:4609', label='MYC', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity',

In [ ]:
def visualize_interaction_network(result_json):
    """
    Visualize the interaction network from the result JSON.

    Parameters:
    result_json (dict): The result JSON containing subjects, objects, and predicates.
    
    """
    import pandas as pd
    import networkx as nx
    import ipycytoscape    
    subjects = []
    objects = []
    predicates = []

    for k, v in result_json.items():
        subjects.append(v['subject'])
        objects.append(v['object'])
        predicates.append(v['predicate'])

    # Convert subject and object CURIEs to names using lookup
    subject_names = []
    object_names = []

    # Use batch_lookup for all subjects and objects to minimize API calls
    unique_nodes = set(subjects + objects)
    node_info_dict = name_resolver.batch_lookup(list(unique_nodes))
    print(node_info_dict)

    for subj, obj in zip(subjects, objects):
        subj_info = node_info_dict.get(subj)
        obj_info = node_info_dict.get(obj)
        subject_names.append(subj_info.name if subj_info and hasattr(subj_info, 'name') else subj)
        object_names.append(obj_info.name if obj_info and hasattr(obj_info, 'name') else obj)


    df = pd.DataFrame({
        "Subject": subject_names,
        "Object": object_names,
        "Predicate": predicates,
    })

    print(df.head())
    # merge predicates and sources
    # Convert 'Sources' to a string representation
    # Remove duplicate edges
    df = df.drop_duplicates()

    # Build networkx graph
    G1 = nx.from_pandas_edgelist(df, source='Subject', target='Object', edge_attr=['Predicate'], create_using=nx.MultiGraph)

    # Cytoscape style
    graph_style = [
        {'selector': 'node', 'style': {'background-color': 'lightblue', 'shape': 'ellipse', 'label': 'data(id)', 'font-size': '12px'}},
        {'selector': 'edge', 'style': {'label': 'data(Predicate)', 'font-size': '10px', "curve-style": "bezier"}}
    ]

    cyto_graph = ipycytoscape.CytoscapeWidget()
    cyto_graph.graph.add_graph_from_networkx(G1, directed=True)
    cyto_graph.set_style(graph_style)
    cyto_graph.set_layout(name='cose', nodeSpacing=80, edgeLengthVal=50)

    display(cyto_graph)

    return df

{'NCBIGene:1050': TranslatorNode(curie='CHEBI:214726', label='1050-Da MAA', types=['biolink:SmallMolecule', 'biolink:MolecularEntity', 'biolink:ChemicalEntity', 'biolink:PhysicalEssence', 'biolink:ChemicalOrDrugOrTreatment', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:ChemicalEntityOrProteinOrPolypeptide', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]), 'NCBIGene:23049': TranslatorNode(curie='PUBCHEM.COMPOUND:89478', label='Wy 23049', types=['biolink:SmallMolecule', 'biolink:MolecularEntity', 'biolink:ChemicalEntity', 'biolink:PhysicalEssence', 'biolink:ChemicalOrDrugOrTreatment', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:ChemicalEntityOrProteinOrPolypeptide', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]), 'NCBIGene:56658': None, 'NCBIGene:8536': TranslatorNode(curie='PUBCHEM.CO

CytoscapeWidget(cytoscape_layout={'name': 'cose', 'nodeSpacing': 80, 'edgeLengthVal': 50}, cytoscape_style=[{'…

,Subject,Object,Predicate
0,NCBIGene:4609,NCBIGene:5595,biolink:associated_with
1,NCBIGene:8880,NCBIGene:4609,biolink:regulates
2,NCBIGene:572,NCBIGene:7157,biolink:regulates
3,NCBIGene:4609,NCBIGene:5594,biolink:associated_with
4,NCBIGene:5443,NCBIGene:1029,biolink:regulates
...,...,...,...
230,NCBIGene:1984,NCBIGene:7157,biolink:regulates
231,NCBIGene:5562,NCBIGene:7157,biolink:regulates
232,NCBIGene:55602,NCBIGene:7157,biolink:regulates
233,NCBIGene:23411,NCBIGene:7157,biolink:regulates


In [5]:
# 5. Filter results by KG
sele_KG_set = set(sele_KG) if sele_KG else None

filtered = {}
for k, v in all_results.items():
    if not isinstance(v, dict):
        continue
    if sele_KG_set:
        match = False
        for src in v.get('sources', []):
            rid = src.get('resource_id', '')
            if rid in sele_KG_set or rid.split(':')[-1] in sele_KG_set:
                match = True
                break
        if not match:
            for attr in v.get('attributes', []):
                if attr.get('attribute_type_id') in ('provided_by', 'knowledge_source'):
                    if attr.get('value') in sele_KG_set:
                        match = True
                        break
        if not match:
            continue
    filtered[k] = v

len(filtered)

189

In [6]:
visualize_interaction_network(filtered)

{'NCBIGene:1050': TranslatorNode(curie='CHEBI:214726', label='1050-Da MAA', types=['biolink:SmallMolecule', 'biolink:MolecularEntity', 'biolink:ChemicalEntity', 'biolink:PhysicalEssence', 'biolink:ChemicalOrDrugOrTreatment', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:ChemicalEntityOrProteinOrPolypeptide', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]), 'NCBIGene:23049': TranslatorNode(curie='PUBCHEM.COMPOUND:89478', label='Wy 23049', types=['biolink:SmallMolecule', 'biolink:MolecularEntity', 'biolink:ChemicalEntity', 'biolink:PhysicalEssence', 'biolink:ChemicalOrDrugOrTreatment', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:ChemicalEntityOrProteinOrPolypeptide', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]), 'NCBIGene:56658': None, 'NCBIGene:6925': TranslatorNode(curie='NCBIGene:1

CytoscapeWidget(cytoscape_layout={'name': 'cose', 'nodeSpacing': 80, 'edgeLengthVal': 50}, cytoscape_style=[{'…

,Subject,Object,Predicate
0,NCBIGene:8880,NCBIGene:4609,biolink:regulates
1,NCBIGene:572,NCBIGene:7157,biolink:regulates
2,NCBIGene:5443,NCBIGene:1029,biolink:regulates
3,NCBIGene:5599,NCBIGene:7157,biolink:regulates
4,NCBIGene:55159,NCBIGene:7157,biolink:regulates
...,...,...,...
184,NCBIGene:8833,NCBIGene:7157,biolink:regulates
185,NCBIGene:1984,NCBIGene:7157,biolink:regulates
186,NCBIGene:5562,NCBIGene:7157,biolink:regulates
187,NCBIGene:55602,NCBIGene:7157,biolink:regulates


In [12]:
import json
with open('filtered.txt', 'w') as file:
    json.dump(filtered, file)

In [10]:
n_hop = 2
from KGBN import gene_names, steiner_tree
def _resolve_curie_names(curies, curie_to_symbol, filtered_results, gene_names_mod, name_resolver):
    """Populate *curie_to_symbol* for every CURIE in *curies* that is not yet mapped."""
    # 1. Extract subject_name / object_name from edge attributes
    for v in filtered_results.values():
        for attr in v.get('attributes', []):
            tid = attr.get('attribute_type_id')
            if tid == 'subject_name' and v['subject'] not in curie_to_symbol:
                curie_to_symbol[v['subject']] = attr['value']
            elif tid == 'object_name' and v['object'] not in curie_to_symbol:
                curie_to_symbol[v['object']] = attr['value']

    # 2. NCBIGene CURIEs → gene symbols via local gene_names module
    for curie in curies:
        if curie not in curie_to_symbol and curie.startswith('NCBIGene:'):
            try:
                curie_to_symbol[curie] = gene_names_mod.get_symbol(int(curie.split(':')[1]))
            except (KeyError, ValueError):
                pass

    # 3. Remaining unknowns → Translator name_resolver
    unknown = [c for c in curies if c not in curie_to_symbol]
    if unknown:
        try:
            extra = name_resolver.batch_lookup(unknown)
            for c in unknown:
                info = extra.get(c)
                if info:
                    name = getattr(info, 'label', None) or getattr(info, 'name', None)
                    if name:
                        curie_to_symbol[c] = name
                        continue
                curie_to_symbol.setdefault(c, c)
        except Exception:
            for c in unknown:
                curie_to_symbol.setdefault(c, c)
                
# 6. Resolve CURIEs → gene symbols
all_curies = set()
for v in filtered.values():
    all_curies.add(v['subject'])
    all_curies.add(v['object'])
_resolve_curie_names(all_curies, curie_to_symbol, filtered, gene_names, name_resolver)

# 7. Collect directed edges as (source_sym, target_sym, pred, quals)
edges_data = []
for k, v in filtered.items():
    s, o = v['subject'], v['object']
    if s == o:
        continue
    sn = curie_to_symbol.get(s, s)
    on = curie_to_symbol.get(o, o)
    edges_data.append((sn, on, v.get('predicate', ''), v.get('qualifiers', [])))
if not edges_data:
    print("No valid directed edges found.")

# 8. For n_hop > 1: build undirected graph, find Steiner tree
if n_hop > 1:
    all_names = sorted({s for s, *_ in edges_data} | {o for _, o, *_ in edges_data})
    G = ig.Graph(directed=False)
    for name in all_names:
        G.add_vertex(name)
    seen_undirected = set()
    for s, o, *_ in edges_data:
        ek = tuple(sorted([s, o]))
        if ek not in seen_undirected:
            seen_undirected.add(ek)
            G.add_edge(s, o)

    input_names = []
    for curie in input_curies:
        sym = curie_to_symbol.get(curie)
        if sym:
            try:
                G.vs.find(name=sym)
                if sym not in input_names:
                    input_names.append(sym)
            except ValueError:
                pass

    if len(input_names) >= 2:
        print(f"Finding Steiner tree for {len(input_names)} input nodes "
                f"in a graph of {len(G.vs)} nodes...")
        tree = steiner_tree.steiner_tree(G, input_names)
        tree_nodes = {v['name'] for v in tree.vs}
        edges_data = [(s, o, p, q) for s, o, p, q in edges_data
                        if s in tree_nodes and o in tree_nodes]
        print(f"Steiner tree: {len(tree_nodes)} nodes, {len(edges_data)} directed edges")

Finding Steiner tree for 3 input nodes in a graph of 180 nodes...
Steiner tree: 3 nodes, 2 directed edges


In [11]:
edges_data

[('CDKN2A',
  'TP53',
  'biolink:regulates',
  [{'qualifier_type_id': 'biolink:object_direction_qualifier',
    'qualifier_value': 'upregulated'}]),
 ('MYC',
  'CDKN2A',
  'biolink:regulates',
  [{'qualifier_type_id': 'biolink:object_direction_qualifier',
    'qualifier_value': 'downregulated'}])]

In [8]:
gene_list = ['CDKN2A', 'MYC', 'TP53']
bn_string, relations = load_translator_network(gene_list, 
        type='gene', 
        joiner='&', 
        n_hop=1,
        # sele_predicate=['biolink:regulates'], 
        sele_API=['CATRAX Pharmacogenomics KP - TRAPI 1.5.0'], 
        sele_KG=['SIGNOR'],
        )
print(bn_string)

Resolving gene names...
Selected tax id: NCBITaxon:9606
info for CDKN2A: TranslatorNode(curie='NCBIGene:1029', label='CDKN2A', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9606'])
info for MYC: TranslatorNode(curie='NCBIGene:4609', label='MYC', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity',

In [14]:
gene_list = ['CDKN2A', 'MYC', 'TP53']
bn_string, relations = load_translator_network(gene_list, 
        type=None, 
        joiner='&', 
        n_hop=1,
        # sele_predicate=['biolink:regulates'], 
        sele_API=['CATRAX Pharmacogenomics KP - TRAPI 1.5.0'], 
        sele_KG=['SIGNOR'],
        )
print(bn_string)

Resolving gene names...
Selected tax id: NCBITaxon:9606
info for CDKN2A: TranslatorNode(curie='NCBIGene:1029', label='CDKN2A', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9606'])
info for MYC: TranslatorNode(curie='NCBIGene:4609', label='MYC', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity',

In [ ]:
gene_list = ['SRSF2', 'MAPK14', 'BAK1', 'ASXL1', 'JAK2', 'U2AF1', 'CBL', 'KRAS', 'STAT3', 'CUX1', 'NFKBIA', 'FLT3', 'PTPN11', 'WT1', 'RUNX1', 'BCL2L1', 'BAX', 'STAG2', 'BCL2L11', 'MCL1', 'NRAS', 'BCOR', 'ATR', 'CEBPA', 'NPM1', 'IDH1', 'BBC3', 'NFKB1', 'MAPK3', 'KIT', 'BAD', 'CALR', 'BCL2', 'DNMT3A', 'IDH2', 'MK167', 'STAT5', 'MAPK1', 'casp3', 'CREBBP', 'EZH2', 'TET2', 'GATA2', 'SF3B1', 'TP53', 'XIAP', 'PHF6', 'CSF3R']
bn_string, relations = load_translator_network(gene_list, 
        type='gene', 
        joiner='&', 
        n_hop=1,
        # sele_predicate=['biolink:regulates'], 
        sele_API=None, 
        sele_KG=['SIGNOR'],
        )
print(bn_string)

Resolving gene names...
info for SRSF2: TranslatorNode(curie='NCBIGene:768117', label='SRSF2', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9823'])
info for MAPK14: TranslatorNode(curie='NCBIGene:100156630', label='MAPK14', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEss

## Combining KG knowledge graphs

In [ ]:
# load the Vundavalli KG
file = './input_files/Vundavilli2020_standardized.txt'
network = KGBN.load_network_from_file(file)
genes = network.nodeDict.keys()
print(f"number of genes: {len(genes)}")
genes

No initial state provided, using a random initial state
Network loaded successfully. There are 38 genes in the network.
number of genes: 38


dict_keys(['EGF', 'HBEGF', 'IGF1', 'NRG1', 'PTEN', 'STK11', 'EGFR', 'ERBB4', 'IGF1R', 'ERBB2', 'JAK1', 'STAT3', 'IRS1', 'GRB2', 'KRAS', 'MAP3K1', 'RAF1', 'MAP2K4', 'MAP2K1', 'PIK3CA', 'MAPK8', 'MAPK3', 'PIP3', 'PDPK1', 'AKT1', 'PRKAA1', 'GSK3B', 'TSC1', 'RHEB', 'MTOR', 'RPS6KB1', 'BAD', 'CCND1', 'BCL2', 'ELK1', 'FOS', 'ELK4', 'SP1'])

In [ ]:
bn_string, relations = KGBN.load_signor_network(genes, joiner='inhibitor_wins', score_cutoff=0.5)
print(bn_string)

Applied score cutoff 0.5, filtered to 19342/40940 edges
number of genes found: 37
[1950, 1839, 3479, 3084, 5728, 6794, 1956, 2066, 3480, 2064, 3716, 6774, 3667, 2885, 3845, 4214, 5894, 6416, 5604, 5290, 5599, 5595, 5170, 207, 5562, 2932, 7248, 6008, 2475, 6198, 572, 595, 596, 2002, 2353, 2005, 5669]
AKT1 = !PTEN & (PDPK1 | MTOR | PIK3CA) # Scores: PDPK1_activate:0.749; MTOR_activate:0.929; PTEN_inhibit:0.634; PIK3CA_activate:0.816
BAD = (!MAPK8 & !AKT1 & !RAF1) # Scores: MAPK8_inhibit:0.686; AKT1_inhibit:0.823; RAF1_inhibit:0.66
BCL2 = (!MAPK8 & !BAD) & MAPK3 # Scores: MAPK8_inhibit:0.581; BAD_inhibit:0.801; MAPK3_activate:0.559
CCND1 = !GSK3B & STAT3 # Scores: GSK3B_inhibit:0.783; STAT3_activate:0.787
EGF = EGF
EGFR = !MAPK3 & (ERBB2 | EGF | HBEGF) # Scores: ERBB2_activate:0.614; MAPK3_inhibit:0.556; EGF_activate:0.949; HBEGF_activate:0.767
ELK1 = (MAPK8 | MAPK3) # Scores: MAPK8_activate:0.512; MAPK3_activate:0.6
ERBB2 = (EGFR | NRG1 | EGF) # Scores: EGFR_activate:0.614; NRG1_activate

In [11]:
KG = KGBN.load_network_from_string(bn_string)
# Merge the networks using inhibitor wins
merged_network_string = KGBN.merge_networks([network, KG], method="Inhibitor Wins", descriptive=True)

No initial state provided, using a random initial state
Network loaded successfully. There are 34 genes in the network.
Merging Method: Inhibitor Wins
Total Genes in Merged Network: 38
Number of Genes in Each Individual Model:
  Model 1: 38 genes
  Model 2: 34 genes
Overlapping Genes: 34
Overlapping Genes List: AKT1, BAD, BCL2, CCND1, EGF, EGFR, ELK1, ERBB2, ERBB4, FOS, GRB2, GSK3B, HBEGF, IGF1, IGF1R, IRS1, JAK1, KRAS, MAP2K1, MAP2K4, MAP3K1, MAPK3, MAPK8, MTOR, NRG1, PDPK1, PIK3CA, PRKAA1, PTEN, RAF1, RPS6KB1, STAT3, STK11, TSC1

Gene: AKT1
  Model 1 Function: PIP3
  Model 2 Function: !PTEN & ( MTOR | PDPK1 | PIK3CA )
  Merged Function: !PTEN & ( MTOR | PDPK1 | PIK3CA | PIP3 )

Gene: BAD
  Model 1 Function: ! ( AKT1 | RPS6KB1 )
  Model 2 Function: !AKT1 & !MAPK8 & !RAF1
  Merged Function: !AKT1 & !MAPK8 & !RAF1 & RPS6KB1

Gene: BCL2
  Model 1 Function: !BAD & STAT3
  Model 2 Function: !BAD & MAPK3 & !MAPK8
  Merged Function: !BAD & !MAPK8 & ( MAPK3 | STAT3 )

Gene: CCND1
  Model 1 Fu

In [13]:
print(merged_network_string)

AKT1 = !PTEN & ( MTOR | PDPK1 | PIK3CA | PIP3 )
BAD = !AKT1 & !MAPK8 & !RAF1 & RPS6KB1
BCL2 = !BAD & !MAPK8 & ( MAPK3 | STAT3 )
CCND1 = !GSK3B & STAT3
EGF = EGF
EGFR = !MAPK3 & ( EGF | ERBB2 | HBEGF )
ELK1 = MAPK3 | MAPK8 | RPS6KB1
ELK4 = MAPK3 & RPS6KB1
ERBB2 = EGF | EGFR | NRG1
ERBB4 = !MAPK3 & ( EGF | ERBB2 | HBEGF | NRG1 )
FOS = MAPK3 | MAPK8 | RPS6KB1
GRB2 = EGFR | ERBB2 | ERBB4 | IGF1R | IRS1
GSK3B = !AKT1
HBEGF = HBEGF
IGF1 = IGF1
IGF1R = IGF1
IRS1 = !MAPK3 & !MAPK8 & !MTOR & !PIK3CA & !PTEN & !RPS6KB1 & ( IGF1R | JAK1 )
JAK1 = EGFR
KRAS = GRB2 | KRAS
MAP2K1 = MAP3K1 | RAF1
MAP2K4 = !AKT1 & MAP3K1
MAP3K1 = KRAS | MAP3K1
MAPK3 = MAP2K1
MAPK8 = MAP2K4
MTOR = !RPS6KB1 & !TSC1 & ( PIK3CA | RHEB )
NRG1 = NRG1
PDPK1 = PIP3 & !RPS6KB1
PIK3CA = !PTEN & ( ERBB2 | ERBB4 | IRS1 | KRAS | STAT3 )
PIP3 = PIK3CA | !PTEN
PRKAA1 = STK11
PTEN = PTEN & !STK11
RAF1 = !AKT1 & !MAPK3 & ( KRAS | MAP2K1 )
RHEB = !TSC1
RPS6KB1 = !PTEN & ( MAPK3 | MTOR | PDPK1 )
SP1 = MAPK3
STAT3 = EGFR | JAK1 | MAPK3 | 

In [ ]:
# Merge the networks using PBN
# Here, a probability of 0.9 is used for rules from the original network
merged_network_string = KGBN.merge_networks([network, KG], method="PBN", prob=0.9)
print(merged_network_string)

AKT1 = !PTEN & ( MTOR | PDPK1 | PIK3CA ), 0.1
AKT1 = PIP3, 0.9
BAD = ! ( AKT1 | RPS6KB1 ), 0.9
BAD = !AKT1 & !MAPK8 & !RAF1, 0.1
BCL2 = !BAD & MAPK3 & !MAPK8, 0.1
BCL2 = !BAD & STAT3, 0.9
CCND1 = !GSK3B & STAT3, 0.1
CCND1 = !GSK3B, 0.9
EGF = EGF, 0.9
EGFR = !MAPK3 & ( EGF | ERBB2 | HBEGF ), 0.1
EGFR = EGF, 0.9
ELK1 = MAPK3 & RPS6KB1, 0.9
ELK1 = MAPK3 | MAPK8, 0.1
ELK4 = MAPK3 & RPS6KB1, 1.0
ERBB2 = EGF | EGFR | NRG1, 0.1
ERBB2 = NRG1, 0.9
ERBB4 = !MAPK3 & ( ERBB2 | HBEGF | NRG1 ), 0.1
ERBB4 = EGF | HBEGF, 0.9
FOS = MAPK3, 0.1
FOS = MAPK8 & RPS6KB1, 0.9
GRB2 = EGFR | ERBB2 | ERBB4 | IGF1R, 0.9
GRB2 = ERBB2 | ERBB4 | IRS1, 0.1
GSK3B = !AKT1, 0.9
HBEGF = HBEGF, 0.9
IGF1 = IGF1, 0.9
IGF1R = IGF1, 0.9
IRS1 = !MAPK3 & !MAPK8 & !MTOR & !PIK3CA & !PTEN & !RPS6KB1 & ( IGF1R | JAK1 ), 0.1
IRS1 = IGF1R, 0.9
JAK1 = EGFR, 0.9
KRAS = GRB2, 0.9
KRAS = KRAS, 0.1
MAP2K1 = MAP3K1 | RAF1, 0.1
MAP2K1 = RAF1, 0.9
MAP2K4 = !AKT1 & MAP3K1, 0.1
MAP2K4 = MAP3K1, 0.9
MAP3K1 = KRAS, 0.9
MAP3K1 = MAP3K1, 0.1
MAPK

## Visualization

In [16]:
pbn = KGBN.load_pbn_from_string(merged_network_string)
KGBN.vis_network(pbn, output_html="Vundavilli2020_extendedPBN.html", interactive=True)

No initial state provided, using a random initial state
PBN loaded successfully. There are 38 genes in the network.
Network visualization saved to Vundavilli2020_extendedPBN.html


# Calculate phenotype score using KG

In [1]:
import pandas as pd
file_path = 'KG_files/significant_paths_to_phenotypes.txt'
df = pd.read_csv(file_path, sep='\t')
print(f'There are {df["EndNode"].nunique()} phenotypes')
print(f'There are {df["QueryNode"].nunique()} genes')

There are 201 phenotypes
There are 4905 genes


In [35]:
print(df["EndNode"].unique())

['ACROSOME_ASSEMBLY' 'ACTIN_CYTOSKELETON_REORGANIZATION'
 'ACTION_POTENTIAL_' 'ADIPOGENESIS' 'ALTERNATIVE_SPLICING_REGULATION'
 'AMYLOID_FIBRIL_FORMATION' 'ANGIOGENESIS' 'APOPTOSIS' 'ARDS'
 'AUTOPHAGOSOME_FORMATION' 'AUTOPHAGY' 'AXONAL_GROWTH_CONE_FORMATION'
 'B_CELL_MATURATION' 'B-LYMPHOCYTE_DIFF' 'BASOPHIL_DIFF'
 'BONE_MINERALIZATION' 'BROWN_ADIPOGENESIS' 'CARTILAGE_DEVELOPMENT'
 'CELL_ADHESION' 'CELL_CYCLE_BLOCK' 'CELL_CYCLE_EXIT'
 'CELL_CYCLE_PROGRESS_' 'CELL_DEATH' 'CELL_GROWTH' 'CELL_KILLING'
 'CELL_MIGRATION' 'CELL_POLARITY' 'CELL_SHAPE' 'CENTROMERE_ASSEMBLY'
 'CENTROSOME_SEPARATION' 'CEREBRAL_CORTEX_DEVELOPMENT'
 'CHAPERONE-MEDIATED_AUTOPHAGY' 'CHAPERONE-MEDIATED_PROTEIN_FOLDING'
 'CHEMOATTRACTION_OF_AXON' 'CHEMOREPULSION_OF_AXON' 'CHEMOTAXIS'
 'CHROMATINE_CONDENSATION' 'CHROMOSOME_SEGREGATION' 'CILIUM_ASSEMBLY'
 'CILIUM_MOVEMENT' 'CITRIC_ACID_CYCLE'
 'CLEARANCE_OF_FOREIGN_INTRACELLULAR_DNA' 'COLLOID' 'CYTOKINE_PRODUCTION'
 'CYTOSKELETON_ORGANIZATION' 'CYTOTOXIC_T-LYMPHOCYTE_AC

In [ ]:
def proxpath(genes, phenotypes = ['APOPTOSIS', 'DIFFERENTIATION', 'PROLIFERATION'], file_path = 'KG_files/significant_paths_to_phenotypes.txt'):
    # Load the ProxPath file
    df = pd.read_csv(file_path, sep='\t')
    
    # Function to find the closest gene to the EndNode
    def closest_gene(path_string, genes):
        # Split the path into components and reverse it (to start from the EndNode)
        components = path_string.split('--')[::-1]
        for component in components:
            # Check if the component contains any of the genes
            for gene in genes:
                if gene in component:
                    return gene
        return None
    
    # Filter rows for the given genes and phenotypes
    filtered_df = df[df['QueryNode'].isin(genes) & df['EndNode'].isin(phenotypes)].copy()
    
    # find the closest gene to the phenotype
    filtered_df['Closest_Gene'] = filtered_df.apply(lambda row: closest_gene(row['Path_String'], genes), axis=1)
    
    # Filter rows where QueryNode is the closest gene to the phenotype
    pheno_df = filtered_df[filtered_df['QueryNode'] == filtered_df['Closest_Gene']]
    pheno_df = pheno_df.drop(columns=['Closest_Gene'])

    # sort
    pheno_df = pheno_df.sort_values(by=['EndNode', 'QueryNode'])

    # remove rows where Final_Effect is 0
    pheno_df = pheno_df[pheno_df['Final_Effect'] != 0]
    
    # save the filtered data to a new file
    # pheno_df.to_csv('Phenotypes.txt', sep='\t', index=False)

    return pheno_df

pheno_df = proxpath(genes = ['KRAS', 'GNAS', 'TP53', 'SMAD4', 'CDKN2A', 'RNF43'], phenotypes = ['PROLIFERATION'])
pheno_df

,EndPathways,QueryNode,EndNode,Path_String,relations_path,Path_Score,Path_Length,Final_Effect,Effect,n,mean,sd,zscore
53815,PROLIFERATION,CDKN2A,PROLIFERATION,CDKN2A--|CDK4--[]CYCLIND/CDK4--|RB1--|PROLIFER...,SIGNOR-44554;SIGNOR-32301;SIGNOR-250762;SIGNOR...,0.577,4,-1,down-regulates,107507,1.681607,0.506006,-2.182991
53913,PROLIFERATION,CDKN2A,PROLIFERATION,CDKN2A--|PROLIFERATION,SIGNOR-259406,0.300,1,-1,down-regulates,107507,1.681607,0.506006,-2.730415
54033,PROLIFERATION,CDKN2A,PROLIFERATION,CDKN2A--|CYCLIND/CDK4--|RB1--|PROLIFERATION,SIGNOR-245459;SIGNOR-250762;SIGNOR-262533,0.633,3,-1,down-regulates,107507,1.681607,0.506006,-2.072320
54152,PROLIFERATION,CDKN2A,PROLIFERATION,CDKN2A--|CDK6--|RB1--|PROLIFERATION,SIGNOR-44557;SIGNOR-135189;SIGNOR-262533,0.672,3,-1,down-regulates,107507,1.681607,0.506006,-1.995246
54306,PROLIFERATION,CDKN2A,PROLIFERATION,CDKN2A--|CDK4--|RB1--|PROLIFERATION,SIGNOR-44554;SIGNOR-200483;SIGNOR-262533,0.464,3,-1,down-regulates,107507,1.681607,0.506006,-2.406308
54518,PROLIFERATION,CDKN2A,PROLIFERATION,CDKN2A--|CDK6--|CDKN1A--|CYCLINE/CDK2-->PROLIF...,SIGNOR-44557;SIGNOR-144832;SIGNOR-245462;SIGNO...,0.651,4,-1,down-regulates,107507,1.681607,0.506006,-2.036748
54124,PROLIFERATION,KRAS,PROLIFERATION,KRAS-->PIK3CA--[]PI3K-->PROLIFERATION,SIGNOR-175204;SIGNOR-255299;SIGNOR-255577,0.450,3,1,up-regulates,107507,1.681607,0.506006,-2.433976
54127,PROLIFERATION,KRAS,PROLIFERATION,KRAS-->PIK3CA-->AKT-->PROLIFERATION,SIGNOR-175204;SIGNOR-244429;SIGNOR-254353,0.600,3,1,up-regulates,107507,1.681607,0.506006,-2.137537
54568,PROLIFERATION,KRAS,PROLIFERATION,KRAS-->PIK3CA--[]PI3K-->AKT-->PROLIFERATION,SIGNOR-175204;SIGNOR-255299;SIGNOR-254950;SIGN...,0.681,4,1,up-regulates,107507,1.681607,0.506006,-1.977460
54569,PROLIFERATION,KRAS,PROLIFERATION,KRAS-->PIK3CA--[]PI3K-->AKT1-->PROLIFERATION,SIGNOR-175204;SIGNOR-255299;SIGNOR-255106;SIGN...,0.681,4,1,up-regulates,107507,1.681607,0.506006,-1.977460


In [30]:
# there may be different Final_Effect values for the same QueryNode and EndNode
for gene in pheno_df['QueryNode'].unique():
    for phenotype in pheno_df['EndNode'].unique():
        if pheno_df[(pheno_df['QueryNode'] == gene) & (pheno_df['EndNode'] == phenotype)]['Final_Effect'].nunique() > 1:
            print(f"{gene} has dual effects on {phenotype}")

# Keep only the rows with the lowest Path_Score for each gene-phenotype pair
pheno_df.loc[pheno_df.groupby(['QueryNode', 'EndNode'])['Path_Score'].idxmin()]

,EndPathways,QueryNode,EndNode,Path_String,relations_path,Path_Score,Path_Length,Final_Effect,Effect,n,mean,sd,zscore
53913,PROLIFERATION,CDKN2A,PROLIFERATION,CDKN2A--|PROLIFERATION,SIGNOR-259406,0.30,1,-1,down-regulates,107507,1.681607,0.506006,-2.730415
54124,PROLIFERATION,KRAS,PROLIFERATION,KRAS-->PIK3CA--[]PI3K-->PROLIFERATION,SIGNOR-175204;SIGNOR-255299;SIGNOR-255577,0.45,3,1,up-regulates,107507,1.681607,0.506006,-2.433976
53529,PROLIFERATION,TP53,PROLIFERATION,TP53--|PROLIFERATION,SIGNOR-255669,0.30,1,-1,down-regulates,107507,1.681607,0.506006,-2.730415


In [37]:
pheno_df = proxpath(genes = ['TP53', 'KRAS', 'MYC', 'SMAD4', 'BCL2', 'BAX'], phenotypes = ['APOPTOSIS'])
pheno_df

,EndPathways,QueryNode,EndNode,Path_String,relations_path,Path_Score,Path_Length,Final_Effect,Effect,n,mean,sd,zscore
2903,APOPTOSIS,BAX,APOPTOSIS,BAX-->APOPTOSIS,SIGNOR-261494,0.300,1,1,up-regulates,75493,1.737625,0.526944,-2.728233
2913,APOPTOSIS,BCL2,APOPTOSIS,BCL2--|APOPTOSIS,SIGNOR-249611,0.300,1,-1,down-regulates,75493,1.737625,0.526944,-2.728233
3437,APOPTOSIS,BCL2,APOPTOSIS,BCL2--|BAK1-->APOPTOSIS,SIGNOR-152980;SIGNOR-261493,0.589,2,-1,down-regulates,75493,1.737625,0.526944,-2.179787
2948,APOPTOSIS,KRAS,APOPTOSIS,KRAS-->PIK3CA-->AKT--|FOXO-->APOPTOSIS,SIGNOR-175204;SIGNOR-244429;SIGNOR-252824;SIGN...,0.698,4,-1,down-regulates,75493,1.737625,0.526944,-1.972934
3343,APOPTOSIS,KRAS,APOPTOSIS,KRAS-->PIK3CA--[]PI3K-->AKT--|APOPTOSIS,SIGNOR-175204;SIGNOR-255299;SIGNOR-254950;SIGN...,0.681,4,-1,down-regulates,75493,1.737625,0.526944,-2.005196
3539,APOPTOSIS,KRAS,APOPTOSIS,KRAS-->PIK3CA-->AKT--|APOPTOSIS,SIGNOR-175204;SIGNOR-244429;SIGNOR-260215,0.600,3,-1,down-regulates,75493,1.737625,0.526944,-2.158912
2894,APOPTOSIS,TP53,APOPTOSIS,TP53-->APOPTOSIS,SIGNOR-255678,0.300,1,1,up-regulates,75493,1.737625,0.526944,-2.728233
3195,APOPTOSIS,TP53,APOPTOSIS,TP53-->BAK1-->APOPTOSIS,SIGNOR-124122;SIGNOR-261493,0.615,2,1,up-regulates,75493,1.737625,0.526944,-2.130446


In [ ]:
def pheno_scores(simulation_results, pheno):
    
    # Keep only the rows with the lowest Path_Score for each gene-phenotype pair
    pheno_unique = pheno.loc[pheno.groupby(['QueryNode', 'EndNode'])['Path_Score'].idxmin()]

    phenotypes = ['APOPTOSIS', 'DIFFERENTIATION', 'PROLIFERATION']    
    # Loop through each phenotype
    for phenotype in phenotypes:
        
        # Filter the 'pheno' dataframe for the current phenotype
        filtered_pheno = pheno_unique[pheno_unique['EndNode'] == phenotype]

        # Loop through each row in the filtered 'pheno' dataframe
        for idx, row in filtered_pheno.iterrows():
            gene = row['QueryNode']
            effect = row['Final_Effect']
            
            # Check if the gene is in the simulation results
            if gene in simulation_results.index:
                # Multiply the gene's simulation result by its effect (1 or -1)
                simulation_results.loc[phenotype] += simulation_results.loc[gene] * effect

    return simulation_results

In [3]:
from KGBN.phenotype_score import phenotype_scores, get_phenotypes
get_phenotypes()

There are 201 phenotypes
There are 4905 genes
Available phenotypes: ['ACROSOME_ASSEMBLY' 'ACTIN_CYTOSKELETON_REORGANIZATION'
 'ACTION_POTENTIAL_' 'ADIPOGENESIS' 'ALTERNATIVE_SPLICING_REGULATION'
 'AMYLOID_FIBRIL_FORMATION' 'ANGIOGENESIS' 'APOPTOSIS' 'ARDS'
 'AUTOPHAGOSOME_FORMATION' 'AUTOPHAGY' 'AXONAL_GROWTH_CONE_FORMATION'
 'B_CELL_MATURATION' 'B-LYMPHOCYTE_DIFF' 'BASOPHIL_DIFF'
 'BONE_MINERALIZATION' 'BROWN_ADIPOGENESIS' 'CARTILAGE_DEVELOPMENT'
 'CELL_ADHESION' 'CELL_CYCLE_BLOCK' 'CELL_CYCLE_EXIT'
 'CELL_CYCLE_PROGRESS_' 'CELL_DEATH' 'CELL_GROWTH' 'CELL_KILLING'
 'CELL_MIGRATION' 'CELL_POLARITY' 'CELL_SHAPE' 'CENTROMERE_ASSEMBLY'
 'CENTROSOME_SEPARATION' 'CEREBRAL_CORTEX_DEVELOPMENT'
 'CHAPERONE-MEDIATED_AUTOPHAGY' 'CHAPERONE-MEDIATED_PROTEIN_FOLDING'
 'CHEMOATTRACTION_OF_AXON' 'CHEMOREPULSION_OF_AXON' 'CHEMOTAXIS'
 'CHROMATINE_CONDENSATION' 'CHROMOSOME_SEGREGATION' 'CILIUM_ASSEMBLY'
 'CILIUM_MOVEMENT' 'CITRIC_ACID_CYCLE'
 'CLEARANCE_OF_FOREIGN_INTRACELLULAR_DNA' 'COLLOID' 'CYTOKINE

In [ ]:
phenotype_scores(
    genes = ['TP53', 'KRAS', 'MYC', 'SMAD4', 'BCL2', 'BAX'], 
    phenotypes = ['APOPTOSIS', 'DIFFERENTIATION', 'PROLIFERATION'], 
    file_path = 'KG_files/significant_paths_to_phenotypes.txt', 
    simulation_results=None
    )

Path found for 3 phenotypes: ['APOPTOSIS' 'PROLIFERATION' 'DIFFERENTIATION']


{'APOPTOSIS': 'BAX -  BCL2 -  KRAS + TP53',
 'PROLIFERATION': 'KRAS + MYC -  TP53',
 'DIFFERENTIATION': 'MYC + TP53'}

In [ ]:
phenotype_scores(
    genes = ['TP53', 'KRAS', 'MYC', 'SMAD4', 'BCL2', 'BAX'], 
    phenotypes = ['APOPTOSIS', 'DIFFERENTIATION', 'PROLIFERATION'], 
    file_path = 'KG_files/significant_paths_to_phenotypes.txt', 
    simulation_results=None
    )

# Test

In [ ]:
import pandas as pd
signor_file = 'KG_files/SIGNOR_2025_08_14.tsv'
graph_table = pd.read_csv(signor_file, index_col=None, sep='\t')
print(graph_table.columns)

Index(['subject_id', 'object_id', 'subject_id_prefix', 'object_id_prefix',
       'subject_name', 'object_name', 'predicate', 'Primary_Knowledge_Source',
       'Knowledge_Source', 'publications', 'subject_category',
       'object_category', 'score'],
      dtype='object')


In [4]:
graph_table['score'].describe()

count    40935.000000
mean         0.501978
std          0.235996
min          0.100000
25%          0.278000
50%          0.468000
75%          0.727000
max          1.000000
Name: score, dtype: float64